# 🚀 Lab 25: Build a COVID Trend Dashboard with Plotly

### 📘 Lab Overview
In this lab, you will build an interactive COVID-19 dashboard using **Plotly** and **Dash**. You will load COVID-style CSV data, preprocess it into a clean country-level time series, create interactive charts for cases, deaths, recoveries, and new cases, and then combine those charts into a dashboard with dropdowns and date controls.

This lab has been adapted for **Google Colab**. Instead of relying on a terminal-only workflow, this notebook builds the app directly in cells. To make the lab reliable, the data loader first tries the archived Johns Hopkins time-series CSV files and falls back to embedded sample CSV data if the network source is unavailable.

---
## 🎯 Objectives
* Import and process COVID-19 dataset data from CSV files
* Create interactive visualizations using Plotly
* Build line charts for confirmed cases, deaths, recoveries, and daily new cases
* Implement dropdown menus and date-range filtering
* Design a multi-chart dashboard layout
* Compare multiple countries in a single interactive chart
* Run a Dash-based web dashboard in a notebook-friendly way
* Export and test dashboard components

---
## 🧰 Prerequisites
* Basic understanding of Python programming
* Familiarity with pandas for data manipulation
* Basic knowledge of data visualization
* General understanding of interactive dashboards

## ⚙️ Environment Setup
### 💡 ELI10: What are we doing?
Before we start, we need to tell Colab to install special tools (libraries) that help us draw charts (Plotly) and build a website-like dashboard (Dash). We also use Pandas to organize our data into tables.

In [1]:
# Install necessary libraries for data manipulation and dashboarding
# %pip is the standard way to install packages in Colab
%pip install plotly dash pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 38.3 MB/s eta 0:00:00


# Task 1: Environment Setup and Data Import

## Subtask 1.1: Import Required Libraries
We need to bring our tools into the notebook workspace.

In [2]:
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output
import datetime as dt
import numpy as np
from io import StringIO

# Note: Modern Dash (2.11+) supports Jupyter/Colab natively.
# We will use the standard dash import and run_server with jupyter_mode.

print("Plotly version:", plotly.__version__)
print("Dash version:", dash.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Plotly version: 5.24.1
Dash version: 4.1.0
Pandas version: 2.2.2
NumPy version: 2.0.2


## 📥 Loading the COVID Data
### 💡 ELI10: What are we doing?
We are asking the computer to go to the internet and grab files that contain COVID numbers. Because these files are old (archived), we also have a 'backup' list of numbers inside our code just in case the internet link doesn't work!

In [3]:
def load_covid_data():
    """
    Load COVID-19 data from archived Johns Hopkins CSV files.
    If remote loading fails, fall back to embedded sample CSV data.
    """
    # URLs for the archived JHU repository
    url_confirmed = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
    url_deaths = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv"
    url_recovered = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_recovered_global.csv"

    try:
        # Attempt to read data directly from GitHub
        df_confirmed = pd.read_csv(url_confirmed)
        df_deaths = pd.read_csv(url_deaths)
        df_recovered = pd.read_csv(url_recovered)

        print("Loaded archived Johns Hopkins COVID-19 data successfully.")
        return df_confirmed, df_deaths, df_recovered

    except Exception as e:
        print(f"Remote load failed: {e}")
        print("Using embedded sample CSV data instead.")

        # Fallback data representing a small subset of countries/dates
        confirmed_csv = StringIO("""Province/State,Country/Region,Lat,Long,1/1/23,1/2/23,1/3/23,1/4/23,1/5/23,1/6/23,1/7/23
,US,37.0902,-95.7129,1000000,1008000,1017000,1025000,1032000,1041000,1050000
,China,35.8617,104.1954,500000,502000,504500,507000,509000,511000,513500
,Italy,41.8719,12.5674,300000,302000,304000,306500,309000,311500,314000
,India,20.5937,78.9629,800000,806000,812000,819000,826000,834000,842000
""")

        deaths_csv = StringIO("""Province/State,Country/Region,Lat,Long,1/1/23,1/2/23,1/3/23,1/4/23,1/5/23,1/6/23,1/7/23
,US,37.0902,-95.7129,12000,12080,12170,12250,12320,12410,12500
,China,35.8617,104.1954,6000,6010,6020,6030,6040,6050,6065
,Italy,41.8719,12.5674,8000,8040,8080,8120,8160,8200,8240
,India,20.5937,78.9629,7000,7040,7080,7130,7180,7240,7300
""")

        recovered_csv = StringIO("""Province/State,Country/Region,Lat,Long,1/1/23,1/2/23,1/3/23,1/4/23,1/5/23,1/6/23,1/7/23
,US,37.0902,-95.7129,900000,907000,914000,921000,928000,936000,944000
,China,35.8617,104.1954,470000,472000,474000,476500,479000,481500,484000
,Italy,41.8719,12.5674,270000,272000,274000,276000,278500,281000,283500
,India,20.5937,78.9629,760000,765000,771000,777000,784000,791000,799000
""")

        df_confirmed = pd.read_csv(confirmed_csv)
        df_deaths = pd.read_csv(deaths_csv)
        df_recovered = pd.read_csv(recovered_csv)

        return df_confirmed, df_deaths, df_recovered

# Execute the loading function
df_confirmed, df_deaths, df_recovered = load_covid_data()

print("Confirmed shape:", df_confirmed.shape)
print("Deaths shape:", df_deaths.shape)
print("Recovered shape:", df_recovered.shape)

Loaded archived Johns Hopkins COVID-19 data successfully.
Confirmed shape: (289, 1147)
Deaths shape: (289, 1147)
Recovered shape: (274, 1147)


## Subtask 1.3: Data Preprocessing
### 💡 ELI10: What are we doing?
The data we got is 'wide' (dates are columns). We need to melt it into a 'long' list (one date per row) so the computer can understand time properly. We also calculate 'New Cases' by looking at the difference between today and yesterday.

In [4]:
def preprocess_data(df_confirmed, df_deaths, df_recovered):
    """
    Clean and preprocess COVID-19 data.
    Converts wide time-series tables into a long country/date format.
    """

    def melt_dataframe(df, value_name):
        # Melt turns date columns into a single 'Date' column for easier plotting
        df_melted = df.melt(
            id_vars=["Province/State", "Country/Region", "Lat", "Long"],
            var_name="Date",
            value_name=value_name
        )
        df_melted["Date"] = pd.to_datetime(df_melted["Date"])
        return df_melted

    # Process the three datasets
    df_confirmed_melted = melt_dataframe(df_confirmed, "Confirmed")
    df_deaths_melted = melt_dataframe(df_deaths, "Deaths")
    df_recovered_melted = melt_dataframe(df_recovered, "Recovered")

    # Merge all datasets on common keys
    df_merged = (
        df_confirmed_melted
        .merge(
            df_deaths_melted[["Province/State", "Country/Region", "Date", "Deaths"]],
            on=["Province/State", "Country/Region", "Date"],
            how="left"
        )
        .merge(
            df_recovered_melted[["Province/State", "Country/Region", "Date", "Recovered"]],
            on=["Province/State", "Country/Region", "Date"],
            how="left"
        )
    )

    # Fill missing values with zero
    df_merged["Deaths"] = df_merged["Deaths"].fillna(0)
    df_merged["Recovered"] = df_merged["Recovered"].fillna(0)

    # Aggregate to country level (removing province/state detail)
    df_country = (
        df_merged
        .groupby(["Country/Region", "Date"], as_index=False)
        .agg({
            "Confirmed": "sum",
            "Deaths": "sum",
            "Recovered": "sum"
        })
    )

    # Sort and calculate daily changes (New Cases = Today - Yesterday)
    df_country = df_country.sort_values(["Country/Region", "Date"])
    df_country["New_Cases"] = df_country.groupby("Country/Region")["Confirmed"].diff().fillna(0)
    df_country["New_Deaths"] = df_country.groupby("Country/Region")["Deaths"].diff().fillna(0)

    # Ensure we don't have negative numbers due to reporting corrections
    df_country["New_Cases"] = df_country["New_Cases"].clip(lower=0)
    df_country["New_Deaths"] = df_country["New_Deaths"].clip(lower=0)

    return df_country

df_processed = preprocess_data(df_confirmed, df_deaths, df_recovered)

print("Processed data preview:")
display(df_processed.head())

print("\nAvailable countries:")
print(sorted(df_processed["Country/Region"].unique()))

/tmp/ipykernel_4179/2184734233.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted["Date"] = pd.to_datetime(df_melted["Date"])
/tmp/ipykernel_4179/2184734233.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted["Date"] = pd.to_datetime(df_melted["Date"])
/tmp/ipykernel_4179/2184734233.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted["Date"] = pd.to_datetime(df_melted["Date"])


Processed data preview:


,Country/Region,Date,Confirmed,Deaths,Recovered,New_Cases,New_Deaths
0,Afghanistan,2020-01-22,0,0,0.0,0.0,0.0
1,Afghanistan,2020-01-23,0,0,0.0,0.0,0.0
2,Afghanistan,2020-01-24,0,0,0.0,0.0,0.0
3,Afghanistan,2020-01-25,0,0,0.0,0.0,0.0
4,Afghanistan,2020-01-26,0,0,0.0,0.0,0.0



Available countries:
['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antarctica', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burma', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo (Brazzaville)', 'Congo (Kinshasa)', 'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Denmark', 'Diamond Princess', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Holy See', 'Honduras', 'Hu

## 📈 Building Trend Charts
### 💡 ELI10: What are we doing?
Now we use Plotly to draw a squiggly line (line chart). When you move your mouse over the line in the notebook, it will tell you the exact number for that day!

In [5]:
def create_trend_chart(df, country, metric="Confirmed"):
    """
    Create interactive line chart for COVID trends using plotly graph objects.
    """
    # Filter data for the specific country
    country_data = df[df["Country/Region"] == country].copy()

    fig = go.Figure()

    # Add the primary data line
    fig.add_trace(go.Scatter(
        x=country_data["Date"],
        y=country_data[metric],
        mode="lines+markers",
        name=f"{metric}",
        line=dict(width=3),
        marker=dict(size=4),
        hovertemplate=(
            "<b>Date:</b> %{x|%Y-%m-%d}<br>"
            f"<b>{metric}:</b> %{{y:,.0f}}<extra></extra>"
        )
    ))

    # Update aesthetics
    fig.update_layout(
        title=f"COVID-19 {metric} Trend - {country}",
        xaxis_title="Date",
        yaxis_title=metric,
        hovermode="x unified",
        template="plotly_white",
        height=500
    )

    return fig

# Test it out for the US
sample_fig = create_trend_chart(df_processed, "US", "Confirmed")
sample_fig.show()

## Subtask 2.2: Create Multi-Metric Dashboard
Let's create 4 charts in one view so we can see everything at once.

In [6]:
def create_multi_metric_chart(df, country):
    """
    Create multi-metric chart with subplots for 4 key indicators.
    """
    country_data = df[df["Country/Region"] == country].copy()

    # Create a 2x2 grid for subplots
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=("Confirmed Cases", "Deaths", "Recovered", "Daily New Cases")
    )

    # 1. Confirmed Cases
    fig.add_trace(go.Scatter(x=country_data["Date"], y=country_data["Confirmed"], name="Confirmed", line=dict(color="blue")), row=1, col=1)

    # 2. Deaths
    fig.add_trace(go.Scatter(x=country_data["Date"], y=country_data["Deaths"], name="Deaths", line=dict(color="red")), row=1, col=2)

    # 3. Recovered
    fig.add_trace(go.Scatter(x=country_data["Date"], y=country_data["Recovered"], name="Recovered", line=dict(color="green")), row=2, col=1)

    # 4. New Cases
    fig.add_trace(go.Scatter(x=country_data["Date"], y=country_data["New_Cases"], name="New Cases", line=dict(color="orange")), row=2, col=2)

    fig.update_layout(
        title_text=f"COVID-19 Dashboard - {country}",
        height=650,
        showlegend=False,
        template="plotly_white"
    )

    return fig

multi_fig = create_multi_metric_chart(df_processed, "US")
multi_fig.show()

## 📊 Building the Dashboard
### 💡 ELI10: What are we doing?
We are building a control panel! We'll add dropdown menus so you can click on a country or a metric, and the charts will magically change to match your choice.

In [10]:
# Initialize Dash app
# In Colab, modern Dash automatically handles the display
app = dash.Dash(__name__)

countries = sorted(df_processed["Country/Region"].unique())

# Define the Layout (What the user sees)
app.layout = html.Div([
    html.H1(
        "COVID-19 Trend Dashboard",
        style={"textAlign": "center", "marginBottom": "30px", "color": "#2c3e50"}
    ),

    html.Div([
        html.Div([
            html.Label("Select Country:", style={"fontWeight": "bold"}),
            dcc.Dropdown(
                id="country-dropdown",
                options=[{"label": country, "value": country} for country in countries],
                value="US"
            )
        ], style={"width": "30%", "display": "inline-block", "marginRight": "3%"}),

        html.Div([
            html.Label("Select Metric:", style={"fontWeight": "bold"}),
            dcc.Dropdown(
                id="metric-dropdown",
                options=[
                    {"label": "Confirmed Cases", "value": "Confirmed"},
                    {"label": "Deaths", "value": "Deaths"},
                    {"label": "Recovered", "value": "Recovered"},
                    {"label": "New Cases", "value": "New_Cases"},
                    {"label": "New Deaths", "value": "New_Deaths"}
                ],
                value="Confirmed"
            )
        ], style={"width": "30%", "display": "inline-block", "marginRight": "3%"}),

        html.Div([
            html.Label("Compare Countries:", style={"fontWeight": "bold"}),
            dcc.Dropdown(
                id="comparison-dropdown",
                options=[{"label": country, "value": country} for country in countries],
                value=["US", "China", "Italy"] if "US" in countries else countries[:3],
                multi=True
            )
        ], style={"width": "30%", "display": "inline-block"})
    ], style={"marginBottom": "20px"}),

    html.Div([
        html.Label("Select Date Range:", style={"fontWeight": "bold"}),
        dcc.DatePickerRange(
            id="date-picker-range",
            start_date=df_processed["Date"].min(),
            end_date=df_processed["Date"].max(),
            display_format="YYYY-MM-DD"
        )
    ], style={"marginBottom": "25px"}),

    dcc.Graph(id="trend-chart"),
    dcc.Graph(id="multi-metric-chart"),
    dcc.Graph(id="comparison-chart")
], style={
    "maxWidth": "1200px",
    "margin": "0 auto",
    "padding": "20px",
    "fontFamily": "Arial, sans-serif"
})

# --- Callbacks (The Logic) ---

@app.callback(
    Output("trend-chart", "figure"),
    Output("multi-metric-chart", "figure"),
    Input("country-dropdown", "value"),
    Input("metric-dropdown", "value"),
    Input("date-picker-range", "start_date"),
    Input("date-picker-range", "end_date")
)
def update_main_charts(selected_country, selected_metric, start_date, end_date):
    # Filter data based on date selection
    filtered_df = df_processed[
        (df_processed["Date" ] >= pd.to_datetime(start_date)) &
        (df_processed["Date" ] <= pd.to_datetime(end_date))
    ]

    trend_fig = create_trend_chart(filtered_df, selected_country, selected_metric)
    multi_fig = create_multi_metric_chart(filtered_df, selected_country)

    return trend_fig, multi_fig

@app.callback(
    Output("comparison-chart", "figure"),
    Input("comparison-dropdown", "value"),
    Input("metric-dropdown", "value"),
    Input("date-picker-range", "start_date"),
    Input("date-picker-range", "end_date")
)
def update_comparison_chart(selected_countries, selected_metric, start_date, end_date):
    filtered_df = df_processed[
        (df_processed["Date" ] >= pd.to_datetime(start_date)) &
        (df_processed["Date" ] <= pd.to_datetime(end_date))
    ]

    fig = go.Figure()
    colors = ["blue", "red", "green", "orange", "purple", "brown", "pink"]

    # Loop through list of countries selected to add multiple lines to one chart
    for i, country in enumerate(selected_countries):
        country_data = filtered_df[filtered_df["Country/Region" ] == country]

        fig.add_trace(go.Scatter(
            x=country_data["Date"],
            y=country_data[selected_metric],
            mode="lines+markers",
            name=country,
            line=dict(color=colors[i % len(colors)], width=3),
            marker=dict(size=4)
        ))

    fig.update_layout(
        title=f"COVID-19 {selected_metric} Comparison",
        xaxis_title="Date",
        yaxis_title=selected_metric,
        hovermode="x unified",
        template="plotly_white",
        height=500,
        legend=dict(x=0.02, y=0.98)
    )

    return fig

# Run the Dashboard inside Colab
# Modern Dash uses .run() instead of .run_server()
# 'jupyter_mode="external"' creates a link to view the dashboard.
app.run(jupyter_mode="external", host="0.0.0.0", port=8050, debug=False)

Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
INFO:werkzeug:Press CTRL+C to quit


## 💾 HTML Export
### 💡 ELI10: What are we doing?
We are saving our cool charts as local files on your computer. This means you can open them in any web browser (like Chrome or Safari) even without running this Python code!

In [11]:
# Export individual figures as standalone HTML
sample_fig.write_html("covid_trend_chart.html")
multi_fig.write_html("covid_multi_metric_dashboard.html")

# Create a combined report manually by injecting JSON data into a template
trend_json = sample_fig.to_json()
multi_json = multi_fig.to_json()

# Note: CSS curly braces must be escaped by doubling them {{ }} to use .format()
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>COVID Trend Dashboard Report</title>
    <script src=\"https://cdn.plot.ly/plotly-latest.min.js\"></script>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1 {{ text-align: center; color: #2c3e50; }}
        h2 {{ color: #34495e; border-bottom: 2px solid #ddd; }}
        .chart-container {{ margin: 30px 0; }}
        .summary {{ background-color: #f8f9fa; padding: 15px; border-radius: 8px; }}
    </style>
</head>
<body>
    <h1>COVID Trend Dashboard Report</h1>
    <div class=\"summary\">
        <h2>Summary</h2>
        <p>This report contains interactive COVID trend visualizations generated with Plotly.</p>
    </div>
    <div class=\"chart-container\">
        <h2>Trend Chart</h2>
        <div id=\"trend-chart\"></div>
    </div>
    <div class=\"chart-container\">
        <h2>Multi-Metric Dashboard</h2>
        <div id=\"multi-chart\"></div>
    </div>
    <script>
        var trendData = {trend_chart_json};
        Plotly.newPlot('trend-chart', trendData.data, trendData.layout);

        var multiData = {multi_chart_json};
        Plotly.newPlot('multi-chart', multiData.data, multiData.layout);
    </script>
</body>
</html>
"""

complete_html = html_template.format(
    trend_chart_json=trend_json,
    multi_chart_json=multi_json
)

with open("covid_dashboard_report.html", "w", encoding="utf-8") as f:
    f.write(complete_html)

print("Created covid_dashboard_report.html and individual HTML components.")

Created covid_dashboard_report.html and individual HTML components.


## ✅ Verification
We need to make sure everything worked correctly.

In [12]:
import glob
import os

# Check if data exists
assert df_processed is not None, "Processed dataframe is missing."
assert len(df_processed) > 0, "Processed dataframe is empty."
print(f"Data loading test passed. Rows loaded: {len(df_processed)}")

# Check if files were created
html_files = sorted(glob.glob("*.html"))
print("\nHTML files found:")
for file_name in html_files:
    print(f"- {file_name} ({os.path.getsize(file_name)} bytes)")

Data loading test passed. Rows loaded: 229743

HTML files found:
- covid_dashboard_report.html (185417 bytes)
- covid_multi_metric_dashboard.html (4701467 bytes)
- covid_trend_chart.html (4601879 bytes)


## 🛠 Troubleshooting

* **Issue: Remote load failed.** This is normal! The Johns Hopkins repository is archived. The code automatically uses sample data if it can't reach GitHub.
* **Issue: Dashboard doesn't appear.** Use `jupyter_mode="external"` to get a link, or try `jupyter_mode="inline"` if you want it to appear inside the cell.
* **Issue: No data for specific dates.** Make sure your Date Range selector covers the dates available in the data (2023 for sample data, or 2020-2023 for real JHU data).

---
## 📚 Key Takeaways / What I Learned
1. **Data Melting:** Learned how to convert wide time-series data into a format suitable for visualization.
2. **Interactive Graphs:** Used Plotly `graph_objects` to create hoverable charts.
3. **Dashboard Callbacks:** Connected dropdown inputs to chart outputs using Dash.
4. **Report Exporting:** Saved interactive findings as portable HTML files.

### Real-World Applications
These skills are used by data analysts to build operational dashboards for business health, logistics tracking, and public health monitoring.